# Fase 3: Núcleo Algorítmico y Programación Orientada a Objetos

## Repositorio y procedencia de los datos

### Repositorio del proyecto

https://github.com/DanielCas7illo/U-Andres-Bello

### Fuente del dataset

https://cotera.co/datasets/library/y-combinator-companies

En esta fase se reorganiza el proceso de limpieza desarrollado en F2 mediante Programación Orientada a Objetos, manteniendo las mismas transformaciones realizadas sobre el conjunto de datos de empresas de Y Combinator.

In [1]:
#verificar versiones
import pandas as pd
import numpy as np
import sys

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.11.5
Pandas: 2.1.1
NumPy: 1.24.3


In [3]:
#Ubicación

from pathlib import Path

def localizar_raiz():
    carpeta_actual = Path.cwd()

    if carpeta_actual.name in {"F1", "F2", "F3", "F4"}:
        return carpeta_actual.parent

    return carpeta_actual

RAIZ = localizar_raiz()

print("Raíz del proyecto:")
print(RAIZ)


Raíz del proyecto:
C:\Users\Casti\U-Andres-Bello\U-Andres-Bello


In [4]:

RUTA_DATASET = "../data/y-combinator-companies.csv"

df_original = pd.read_csv(RUTA_DATASET)
#copia de dataframe para no sobreescribir el original
df = df_original.copy()
df.head()

,company_id,name,yc_slug,yc_url,batch,batch_season,batch_year,batch_code,years_since_batch,status,...,website,domain,domain_tld,website_status,website_live,website_final_domain,website_redirects_offsite,checked_at,batch_size,batch_active_rate
0,32165,Rote,rote,https://www.ycombinator.com/companies/rote,Winter 2027,Winter,2027.0,W27,-1.0,Active,...,https://tryrote.com/,tryrote.com,tryrote.com,200.0,True,tryrote.com,False,2026-08-27,1,1.0
1,35290,Antropi Robotics,antropi-robotics,https://www.ycombinator.com/companies/antropi-...,Fall 2026,Fall,2026.0,F26,0.0,Active,...,https://www.antropi.world/,antropi.world,antropi.world,200.0,True,antropi.world,False,2026-08-27,23,1.0
2,35058,Capveon,capveon,https://www.ycombinator.com/companies/capveon,Fall 2026,Fall,2026.0,F26,0.0,Active,...,https://capveon.ai,capveon.ai,capveon.ai,200.0,True,capveon.ai,False,2026-08-27,23,1.0
3,31461,Collar,collar,https://www.ycombinator.com/companies/collar,Fall 2026,Fall,2026.0,F26,0.0,Active,...,https://usecollarai.com,usecollarai.com,usecollarai.com,200.0,True,usecollarai.com,False,2026-08-27,23,1.0
4,36084,Covera,covera,https://www.ycombinator.com/companies/covera,Fall 2026,Fall,2026.0,F26,0.0,Active,...,https://www.covera-agents.com/,covera-agents.com,covera-agents.com,200.0,True,covera-agents.com,False,2026-08-27,23,1.0


# Selección de variables
Se utilizan las mismas variables definidas durante la Fase 2 para mantener la consistencia del análisis.

In [5]:
variables_analisis = [
    "name",
    "is_operating",
    "is_hiring",
    "nonprofit",
    "status",
    "is_ai",
    "team_size",
    "industry",
    "hq_country",
    "years_since_batch"
]

df = df_original[variables_analisis].copy()

df.head()

,name,is_operating,is_hiring,nonprofit,status,is_ai,team_size,industry,hq_country,years_since_batch
0,Rote,True,False,False,Active,False,0.0,B2B,USA,-1.0
1,Antropi Robotics,True,False,False,Active,False,0.0,Industrials,India,0.0
2,Capveon,True,False,False,Active,False,2.0,Government,USA,0.0
3,Collar,True,False,False,Active,True,2.0,B2B,Singapore,0.0
4,Covera,True,True,False,Active,False,2.0,Fintech,USA,0.0


# Programación Orientada a Objetos
Durante la Fase 2 las transformaciones fueron implementadas mediante funciones.

En esta fase se reorganiza el proceso utilizando una clase, permitiendo encapsular las operaciones de limpieza y reutilizar el código de manera más eficiente.

# Limpieza de valores nulos y Dato Sucio Ycombinator

In [6]:
#Se almacena una copia del DF original para evitar modificaciones accidentales
class PreprocesadorStartups:

    def __init__(self, datos):

        self.df = datos.copy()
#Se elimina de las filas la empresa Ycombinator, dado que son los dueños del dataset,
# La idea es que no ensucie el conjunto de datos
    def eliminar_y_combinator(self):

        filas_antes = len(self.df)

        self.df = self.df[
            self.df["name"] != "Y Combinator"
        ]

        filas_despues = len(self.df)

        print(
            f"Registros eliminados: {filas_antes - filas_despues}"
        )

        return self
#Se cambia tamaño de equipo por la mediana
    def imputar_team_size(self):

        mediana = self.df["team_size"].median()

        nulos = self.df["team_size"].isna().sum()

        self.df["team_size"] = (
            self.df["team_size"]
            .fillna(mediana)
        )

        print(
            f"Nulos reemplazados: {nulos}"
        )

        print(
            f"Mediana utilizada: {mediana}"
        )

        return self
#Se cambian valores vacíos de Países y se cambian por desconocidos
    def imputar_pais(self):

        nulos = self.df["hq_country"].isna().sum()

        self.df["hq_country"] = (
            self.df["hq_country"]
            .fillna("Desconocido")
        )

        print(
            f"Paises reemplazados: {nulos}"
        )

        return self
#en variable years_since_batch(años desde que la startup partició en programa Ycombinator), tenía valor negativo
#Se cambia por cero, era negativo porque la startup fue seleccionada pero empieza en 2027
    def corregir_years(self):

        negativos = (
            self.df["years_since_batch"] < 0
        ).sum()

        self.df.loc[
            self.df["years_since_batch"] < 0,
            "years_since_batch"
        ] = 0

        print(
            f"Valores negativos corregidos: {negativos}"
        )

        return self

    def procesar(self):

        (
            self
            .eliminar_y_combinator()
            .imputar_team_size()
            .imputar_pais()
            .corregir_years()
        )

        return self.df

In [7]:
preprocesador = PreprocesadorStartups(df)

df_procesado = preprocesador.procesar()

df_procesado.head()

Registros eliminados: 1
Nulos reemplazados: 108
Mediana utilizada: 6.0
Paises reemplazados: 154
Valores negativos corregidos: 1


,name,is_operating,is_hiring,nonprofit,status,is_ai,team_size,industry,hq_country,years_since_batch
0,Rote,True,False,False,Active,False,0.0,B2B,USA,0.0
1,Antropi Robotics,True,False,False,Active,False,0.0,Industrials,India,0.0
2,Capveon,True,False,False,Active,False,2.0,Government,USA,0.0
3,Collar,True,False,False,Active,True,2.0,B2B,Singapore,0.0
4,Covera,True,True,False,Active,False,2.0,Fintech,USA,0.0


# Validación Técnica

A continuación se verifica que las transformaciones fueron aplicadas correctamente

In [8]:
assert (
    df_procesado["team_size"].isna().sum() == 0
)

assert (
    df_procesado["hq_country"].isna().sum() == 0
)

assert (
    (df_procesado["years_since_batch"] < 0).sum() == 0
)

assert (
    "Y Combinator"
    not in df_procesado["name"].values
)

print("Validaciones superadas")

Validaciones superadas


# Comparación Entre Fase 2 y Fase 3

La Fase 3 reorganiza mediante Programación Orientada a Objetos las mismas transformaciones realizadas en la Fase 2.

Para verificar que la reorganización no cambió los datos, se ejecuta primero la versión funcional de F2 y después la versión orientada a objetos de F3. Finalmente, ambos resultados se comparan mediante assert_frame_equal.

In [15]:
# Versión funcional utilizada en F2
def limpiar_fase2(datos):

    resultado = datos.copy()

    # Eliminar el registro que no representa una startup
    resultado = resultado[
        resultado["name"] != "Y Combinator"
    ].copy()

    # Imputar team_size con la mediana
    mediana_team = resultado["team_size"].median()

    resultado["team_size"] = (
        resultado["team_size"]
        .fillna(mediana_team)
    )

    # Completar datos vaciós país y cambiarlo por desconocido
    resultado["hq_country"] = (
        resultado["hq_country"]
        .fillna("Desconocido")
    )

    # Corregir años negativos
    resultado.loc[
        resultado["years_since_batch"] < 0,
        "years_since_batch"
    ] = 0

    return resultado.reset_index(drop=True)


# Resultado de la Fase 2
resultado_f2 = limpiar_fase2(df)


# Resultado de la Fase 3
preprocesador = PreprocesadorStartups(df)

resultado_f3 = (
    preprocesador
    .procesar()
    .reset_index(drop=True)
)


# Comparación exacta
pd.testing.assert_frame_equal(
    resultado_f2,
    resultado_f3,
    check_dtype=False
)

print("F2 y F3 producen el mismo resultado.")


Registros eliminados: 1
Nulos reemplazados: 108
Mediana utilizada: 6.0
Paises reemplazados: 154
Valores negativos corregidos: 1
F2 y F3 producen el mismo resultado.


# Eficiencia y Optimización

Se comparan dos implementaciones para reemplazar por cero los valores negativos de years_since_batch.

La primera recorre los registros mediante un ciclo for. La segunda utiliza la operación vectorizada mask() de Pandas.

Ambas implementaciones deben entregar el mismo resultado. Posteriormente se mide el tiempo de ejecución y la memoria máxima utilizada en distintos tamaños de entrada.

In [16]:
import timeit
import tracemalloc
import numpy as np
import pandas as pd
#Corrección de negativos utilizando FOR

def corregir_negativos_iterativo(serie):
    """Corrección mediante recorrido elemento por elemento."""

    resultado = serie.copy()

    for posicion in range(len(resultado)):

        if resultado.iloc[posicion] < 0:

            resultado.iloc[posicion] = 0

    return resultado

#Corrección utilizando vectorizado

def corregir_negativos_vectorizado(serie):
    """Corrección mediante una operación vectorizada de Pandas."""

    return serie.mask(serie < 0, 0)

# Verificación de Equivalencia

In [17]:
# Se eliminan los nulos porque estos registros ya son tratados
# anteriormente por el pipeline de limpieza.

verificacion_prueba = (
    df["years_since_batch"]
    .dropna()
    .reset_index(drop=True)
)

resultado_iterativo = corregir_negativos_iterativo(
    verificacion_prueba
)

resultado_vectorizado = corregir_negativos_vectorizado(
    verificacion_prueba
)

pd.testing.assert_series_equal(
    resultado_iterativo,
    resultado_vectorizado
)

print("Ambas implementaciones producen el mismo resultado.")

Ambas implementaciones producen el mismo resultado.


# Medición Tiempo y Memoria

In [21]:
def medir_rendimiento(funcion, serie, repeticiones=10):
    
    #Mide el tiempo promedio y la memoria máxima utilizada por una función.
    

    tiempo_total = timeit.timeit(
        lambda: funcion(serie),
        number=repeticiones
    )

    tiempo_promedio = tiempo_total / repeticiones

    tracemalloc.start()

    funcion(serie)

    _, memoria_maxima = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    return tiempo_promedio, memoria_maxima / 1024

In [23]:
# En la siguiente celda se prueban distintos tamaños

tamanos = [1_000, 10_000, 100_000]

resultados_eficiencia = []

valores_base = verificacion_prueba.to_numpy()

for tamano in tamanos:

    # Repite los datos hasta alcanzar el tamaño solicitado
    datos_ampliados = pd.Series(
        np.resize(valores_base, tamano)
    )

    tiempo_iterativo, memoria_iterativa = medir_rendimiento(
        corregir_negativos_iterativo,
        datos_ampliados
    )

    tiempo_vectorizado, memoria_vectorizada = medir_rendimiento(
        corregir_negativos_vectorizado,
        datos_ampliados
    )

    resultados_eficiencia.append({
        "tamaño": tamano,
        "tiempo_iterativo_s": tiempo_iterativo,
        "tiempo_vectorizado_s": tiempo_vectorizado,
        "memoria_iterativa_kb": memoria_iterativa,
        "memoria_vectorizada_kb": memoria_vectorizada
    })

tabla_eficiencia = pd.DataFrame(resultados_eficiencia)

tabla_eficiencia

,tamaño,tiempo_iterativo_s,tiempo_vectorizado_s,memoria_iterativa_kb,memoria_vectorizada_kb
0,1000,0.006198,0.000399,9.995117,17.636719
1,10000,0.055200,0.000267,80.338867,131.894531
2,100000,0.563534,0.000546,783.581055,1274.472656
